# Claude Code Built-in Tools 분석

## 실험 목적

Claude Code의 주요 내장 툴이 Anthropic API 레벨에서 어떤 형태로 호출되는지 관찰하고, 각 툴의 **토큰 소비량**을 비교 분석한다.

| Section | 내용 |
|---------|------|
| 0 | 프록시 상태 확인 및 클라이언트 초기화 |
| 1 | Tool Schema 정의 & 토큰 비용 측정 (근사치) |
| 2 | 실제 tool_use 호출 & JSON 구조 관찰 |
| 3 | 툴별 토큰 소비량 비교 Figure |
| 4 | Claude Code 실제 Schema 분석 (프록시 캡처) |

## 실행 시나리오

### Section 1~3만 실행 (프록시 불필요)
```
Kernel > Restart & Run All
```

### Section 4까지 실행 (Claude Code 실제 schema 캡처)
```
# Phase 1 — 캡처 준비 (별도 터미널, 한 번만)
cd deepagent-builtin-tools-analysis
bash script/start_proxy.sh          # 프록시 백그라운드 실행

# .env에 추가 후 Claude Code 재시작
ANTHROPIC_BASE_URL=http://localhost:8082

# Phase 2 — 분석 (Claude Code가 툴 사용 후 언제든)
Kernel > Restart & Run All
```

In [12]:
import os
import json
import socket
import anthropic
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
MODEL = 'claude-haiku-4-5-20251001'
plt.rcParams['figure.dpi'] = 130
print(f'Model: {MODEL}')

Model: claude-haiku-4-5-20251001


---
## Section 0: 프록시 상태 확인 & 클라이언트 초기화

프록시가 실행 중이면 모든 API 호출을 프록시를 통해 기록한다.  
프록시가 없어도 Section 1~3은 정상 실행된다.

In [ ]:
PROXY_PORT = 8082
PROXY_BASE  = f'http://localhost:{PROXY_PORT}'

def _port_open(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        return s.connect_ex(('localhost', port)) == 0

PROXY_RUNNING = _port_open(PROXY_PORT)

if PROXY_RUNNING:
    # 모든 API 호출이 프록시를 통해 JSONL에 기록됨
    client = anthropic.Anthropic(
        api_key=os.environ['ANTHROPIC_API_KEY'],
        base_url=PROXY_BASE
    )
    print(f'[Proxy] RUNNING  → {PROXY_BASE}')
    print(f'        logging  → dataset/captured_requests.jsonl')
else:
    # 프록시 없음 — Section 1~3은 정상 실행, Section 4는 건너뜀
    client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
    print('[Proxy] NOT running  (Section 1~3 정상 실행 | Section 4 스킵)')
    print()
    print('  프록시 시작:  bash script/start_proxy.sh')
    print('  Claude Code 재시작 전 .env에 추가:')
    print('  ANTHROPIC_BASE_URL=http://localhost:8082')

# count_tokens는 항상 직접 연결 사용 (프록시 로그 오염 방지)
client_direct = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])

---
## Section 1: Tool Schema 정의 & 토큰 비용 측정

각 툴의 `description` + `input_schema`를 정의하고, 스키마 자체가 소비하는 입력 토큰 수를 `count_tokens`으로 측정한다.

> ⚠️ 이 섹션의 스키마는 공개 문서 기반 **근사치**입니다. 실제 Claude Code 스키마는 Section 4에서 확인합니다.

In [ ]:
TOOLS = [
    {
        'name': 'Bash',
        'description': 'Executes a bash command and returns its output. The working directory persists between commands.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'command':           {'type': 'string',  'description': 'The command to execute'},
                'description':       {'type': 'string',  'description': 'What this command does'},
                'timeout':           {'type': 'integer', 'description': 'Optional timeout in milliseconds (max 600000)'},
                'run_in_background': {'type': 'boolean', 'description': 'Run in background'}
            },
            'required': ['command', 'description']
        }
    },
    {
        'name': 'Read',
        'description': 'Reads a file from the local filesystem. Returns content with line numbers in cat -n format.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'file_path': {'type': 'string',  'description': 'Absolute path to the file to read'},
                'limit':     {'type': 'integer', 'description': 'Number of lines to read'},
                'offset':    {'type': 'integer', 'description': 'Line number to start reading from'},
                'pages':     {'type': 'string',  'description': 'Page range for PDF files'}
            },
            'required': ['file_path']
        }
    },
    {
        'name': 'Write',
        'description': 'Writes content to a file on the local filesystem. Overwrites the existing file if one exists.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'file_path': {'type': 'string', 'description': 'Absolute path to the file to write'},
                'content':   {'type': 'string', 'description': 'The content to write to the file'}
            },
            'required': ['file_path', 'content']
        }
    },
    {
        'name': 'Edit',
        'description': 'Performs exact string replacements in files. The edit fails if old_string is not unique.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'file_path':   {'type': 'string',  'description': 'Absolute path to the file to modify'},
                'old_string':  {'type': 'string',  'description': 'The text to replace'},
                'new_string':  {'type': 'string',  'description': 'The text to replace it with'},
                'replace_all': {'type': 'boolean', 'default': False, 'description': 'Replace all occurrences'}
            },
            'required': ['file_path', 'old_string', 'new_string']
        }
    },
    {
        'name': 'WebSearch',
        'description': 'Searches the web for real-time information using a search query.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'query':           {'type': 'string', 'description': 'The search query'},
                'allowed_domains': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Domains to restrict search to'},
                'blocked_domains': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Domains to block'}
            },
            'required': ['query']
        }
    },
    {
        'name': 'TodoWrite',
        'description': 'Creates and manages a structured todo list for tracking task progress in the current session.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'todos': {
                    'type': 'array',
                    'description': 'The list of todos to write',
                    'items': {
                        'type': 'object',
                        'properties': {
                            'id':       {'type': 'string'},
                            'content':  {'type': 'string'},
                            'status':   {'type': 'string', 'enum': ['pending', 'in_progress', 'completed']},
                            'priority': {'type': 'string', 'enum': ['high', 'medium', 'low']}
                        },
                        'required': ['id', 'content', 'status', 'priority']
                    }
                }
            },
            'required': ['todos']
        }
    }
]

tool_names = [t['name'] for t in TOOLS]
print('Defined tools:', tool_names)

In [ ]:
dummy_msg = [{'role': 'user', 'content': 'x'}]
baseline = client_direct.messages.count_tokens(model=MODEL, messages=dummy_msg).input_tokens

schema_only = {}
for tool in TOOLS:
    total = client_direct.messages.count_tokens(
        model=MODEL, tools=[tool], messages=dummy_msg
    ).input_tokens
    schema_only[tool['name']] = total - baseline

df_schema = pd.DataFrame(list(schema_only.items()), columns=['Tool', 'Schema Tokens'])
print(f'Baseline (no tools): {baseline} tokens\n')
print(df_schema.to_string(index=False))

In [ ]:
COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3', '#937860']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(df_schema['Tool'], df_schema['Schema Tokens'], color=COLORS, edgecolor='white', linewidth=0.8)
ax.bar_label(bars, fmt='%d', padding=3, fontsize=10)
ax.set_title('Tool Schema Definition — Input Token Cost (근사치)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Tool')
ax.set_ylabel('Schema-only Input Tokens')
ax.set_ylim(0, df_schema['Schema Tokens'].max() * 1.25)
ax.grid(axis='y', alpha=0.3)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig('schema_token_cost.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 2: 실제 tool_use 호출 & JSON 구조 관찰

`tool_choice`로 각 툴을 강제 호출하여 `tool_use` 블록의 실제 JSON 구조를 확인한다.  
프록시가 실행 중이면 이 호출들도 `dataset/captured_requests.jsonl`에 기록된다.

In [ ]:
SCENARIOS = {
    'Bash':      'list all Python files in the current directory',
    'Read':      'read the file /etc/hosts',
    'Write':     'write Hello World to /tmp/hello.txt',
    'Edit':      'in /tmp/hello.txt replace Hello with Hi',
    'WebSearch': 'search for Anthropic Claude API pricing 2025',
    'TodoWrite': 'create a todo list: data collection, preprocessing, model training',
}

tool_map  = {t['name']: t for t in TOOLS}
call_data = {}

print(f"{'Tool':<12}  {'in_tok':>6}  {'out_tok':>7}")
print('-' * 30)
for tool_name, prompt in SCENARIOS.items():
    resp = client.messages.create(
        model=MODEL,
        max_tokens=512,
        tools=[tool_map[tool_name]],
        tool_choice={'type': 'tool', 'name': tool_name},
        messages=[{'role': 'user', 'content': prompt}]
    )
    block = next((b for b in resp.content if b.type == 'tool_use'), None)
    call_data[tool_name] = {
        'input_tokens':  resp.usage.input_tokens,
        'output_tokens': resp.usage.output_tokens,
        'block': block
    }
    print(f'{tool_name:<12}  {resp.usage.input_tokens:>6}  {resp.usage.output_tokens:>7}')

In [ ]:
for tool_name, data in call_data.items():
    block = data['block']
    if block is None:
        continue
    sep = '=' * 60
    print(f'\n{sep}')
    print(f'  {tool_name}')
    print(sep)
    print(json.dumps(
        {'type': 'tool_use', 'id': block.id[:24] + '...', 'name': block.name, 'input': block.input},
        indent=2, ensure_ascii=False
    ))

---
## Section 3: 툴별 토큰 소비량 비교 Figure

- **Left**: Input vs Output tokens (grouped bar)
- **Right**: Input token 구성 분해 — Baseline / Schema / Call Overhead (stacked bar)

In [ ]:
df_calls = pd.DataFrame([
    {'Tool': name, 'Input': data['input_tokens'],
     'Output': data['output_tokens'], 'Schema': schema_only.get(name, 0)}
    for name, data in call_data.items()
])
df_calls['Overhead'] = (df_calls['Input'] - df_calls['Schema'] - baseline).clip(lower=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(df_calls))
w = 0.35
b1 = axes[0].bar(x-w/2, df_calls['Input'],  w, label='Input',  color='#4C72B0', alpha=0.85)
b2 = axes[0].bar(x+w/2, df_calls['Output'], w, label='Output', color='#DD8452', alpha=0.85)
axes[0].bar_label(b1, fmt='%d', padding=2, fontsize=8)
axes[0].bar_label(b2, fmt='%d', padding=2, fontsize=8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(df_calls['Tool'], rotation=20, ha='right')
axes[0].set_title('Input vs Output Tokens per Tool Call', fontweight='bold')
axes[0].set_ylabel('Tokens')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
for spine in ['top', 'right']: axes[0].spines[spine].set_visible(False)

base_arr   = np.full(len(df_calls), baseline)
schema_arr = df_calls['Schema'].values
over_arr   = df_calls['Overhead'].values
axes[1].bar(df_calls['Tool'], base_arr,   color='#aec7e8', label='Baseline')
axes[1].bar(df_calls['Tool'], schema_arr, bottom=base_arr,              color='#4C72B0', label='Schema Def')
axes[1].bar(df_calls['Tool'], over_arr,   bottom=base_arr+schema_arr,   color='#ffbb78', label='Call Overhead')
axes[1].set_xticks(range(len(df_calls)))
axes[1].set_xticklabels(df_calls['Tool'], rotation=20, ha='right')
axes[1].set_title('Input Token Breakdown per Tool', fontweight='bold')
axes[1].set_ylabel('Input Tokens')
axes[1].legend(fontsize=8)
axes[1].grid(axis='y', alpha=0.3)
for spine in ['top', 'right']: axes[1].spines[spine].set_visible(False)

plt.suptitle('Claude Code Built-in Tools — Token Consumption Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('tool_token_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df_summary = df_calls[['Tool', 'Schema', 'Input', 'Output']].copy()
df_summary['Total'] = df_summary['Input'] + df_summary['Output']
df_summary = df_summary.sort_values('Total', ascending=False).reset_index(drop=True)
df_summary.columns = ['Tool', 'Schema Tokens', 'Input Tokens', 'Output Tokens', 'Total Tokens']
print('=== Tool Token Consumption Summary ===\n')
print(df_summary.to_string(index=False))
mean_in, mean_out = df_summary['Input Tokens'].mean(), df_summary['Output Tokens'].mean()
print(f'\nMean Input : {mean_in:.1f} tokens')
print(f'Mean Output: {mean_out:.1f} tokens')

---
## Section 4: Claude Code 실제 Schema 분석

프록시가 캡처한 `dataset/captured_requests.jsonl`을 읽어
Claude Code가 실제로 사용하는 tool schema를 분석하고 Section 1의 근사치와 비교한다.

> 이 섹션은 **프록시가 실행 중인 상태에서 Claude Code를 재시작한 뒤**
> Claude Code가 어떤 툴이든 사용해야 캡처 데이터가 쌓입니다.

In [13]:
log_path = Path('dataset/captured_requests.jsonl')

if not log_path.exists():
    print('캡처 데이터 없음.')
    print()
    print('준비 순서:')
    print('  1. bash script/start_proxy.sh   (별도 터미널)')
    print('  2. .env에 ANTHROPIC_BASE_URL=http://localhost:8082 추가')
    print('  3. Claude Code 재시작')
    print('  4. Claude Code가 어떤 툴이든 사용 (자동 캡처)')
    print('  5. 이 셀 재실행')
    cc_captures = []
else:
    cc_captures = []
    with open(log_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                cc_captures.append(json.loads(line))

    with_tools = [c for c in cc_captures if c['tool_count'] > 0]
    print(f'총 캡처 요청: {len(cc_captures)}  |  tool 포함: {len(with_tools)}')

    if with_tools:
        best = max(with_tools, key=lambda c: c['tool_count'])
        print(f'\n최다 tool 캡처: model={best["model"]}, tool_count={best["tool_count"]}')
        print('tool names:', [t['name'] for t in best['tools']])

캡처 데이터 없음.

준비 순서:
  1. bash script/start_proxy.sh   (별도 터미널)
  2. .env에 ANTHROPIC_BASE_URL=http://localhost:8082 추가
  3. Claude Code 재시작
  4. Claude Code가 어떤 툴이든 사용 (자동 캡처)
  5. 이 셀 재실행


In [ ]:
if not cc_captures or not any(c['tool_count'] > 0 for c in cc_captures):
    print('캡처 데이터가 없어 비교를 건너뜁니다.')
else:
    best   = max(cc_captures, key=lambda c: c['tool_count'])
    cc_map = {t['name']: t for t in best['tools']}
    our_map = {t['name']: t for t in TOOLS}
    common  = sorted(set(cc_map) & set(our_map))
    cc_only = sorted(set(cc_map) - set(our_map))

    print(f'Claude Code 정의 툴: {len(cc_map)}개')
    print(f'우리 정의 툴        : {len(our_map)}개')
    print(f'공통                : {common}')
    print(f'Claude Code 전용    : {cc_only}')

    if common:
        our_sz = [len(json.dumps(our_map[t])) for t in common]
        cc_sz  = [len(json.dumps(cc_map[t]))  for t in common]
        fig, ax = plt.subplots(figsize=(10, 4))
        x = np.arange(len(common))
        w = 0.35
        b1 = ax.bar(x-w/2, our_sz, w, label='Our Definition',     color='#4C72B0', alpha=0.85)
        b2 = ax.bar(x+w/2, cc_sz,  w, label='Claude Code Actual', color='#DD8452', alpha=0.85)
        ax.bar_label(b1, fmt='%d', padding=2, fontsize=8)
        ax.bar_label(b2, fmt='%d', padding=2, fontsize=8)
        ax.set_xticks(x)
        ax.set_xticklabels(common, rotation=15, ha='right')
        ax.set_title('Schema JSON Size: Our Definition vs Claude Code Actual (chars)', fontweight='bold')
        ax.set_ylabel('Schema JSON size (chars)')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        for spine in ['top', 'right']: ax.spines[spine].set_visible(False)
        plt.tight_layout()
        plt.savefig('schema_size_comparison.png', dpi=150, bbox_inches='tight')
        plt.show()

In [ ]:
if cc_captures and any(c['tool_count'] > 0 for c in cc_captures):
    best = max(cc_captures, key=lambda c: c['tool_count'])
    rows = []
    for t in best['tools']:
        props    = t.get('input_schema', {}).get('properties', {})
        required = t.get('input_schema', {}).get('required', [])
        rows.append({
            'Tool':                 t['name'],
            'Required':             ', '.join(required),
            'Optional Fields':      len(props) - len(required),
            'Schema Size (chars)':  len(json.dumps(t))
        })
    df_cc = pd.DataFrame(rows).sort_values('Schema Size (chars)', ascending=False).reset_index(drop=True)
    print(f'=== Claude Code 실제 Tool 목록 ({len(df_cc)}개) ===\n')
    print(df_cc.to_string(index=False))